# ❄️ Aprofundamento I: Fundamentos da Plataforma Snowflake
**Snowflake Partner Enablement — Lisboa 2026**

---

🎯 **Objetivo:** Demonstrar as capacidades core do Snowflake que resolvem problemas reais dos clientes.

⏱️ **Duração:** ~35 minutos

| # | Secção | Tempo | Conceito-chave |
|---|--------|-------|----------------|
| 1 | 🏗️ Setup do Ambiente | 3 min | Separação compute/storage |
| 2 | 📦 Carga de Dados | 5 min | Formatos, tipos, escala |
| 3 | 🔄 Snowpipe | 7 min | Ingestão contínua sem orquestração |
| 4 | 🔒 Governança | 10 min | Mesma query, dados diferentes |
| 5 | ⚡ Performance & Custos | 10 min | Pay-per-use, caching |

---

💡 **Como usar este notebook:** Execute cada célula sequencialmente. As explicações entre as células descrevem os conceitos-chave e o valor de negócio de cada funcionalidade.

---
## 1. 🏗️ Setup do Ambiente

💡 **Conceito-chave:** A arquitetura do Snowflake separa completamente **compute** de **storage**. Isto significa que podemos escalar cada camada independentemente.

🔑 **Porquê isto importa:** No Snowflake, toda a infraestrutura é criada instantaneamente via SQL — sem tickets de IT, sem aprovações, sem espera. Os clientes reduzem o time-to-value de semanas para segundos.

```
┌─────────────────────────────────────────────────────┐
│  Cloud Services (Metadata, Otimização, Segurança)   │
├─────────────────┬───────────────────────────────────┤
│  Warehouse XS   │   Warehouse MEDIUM                │
│  (Analistas)    │   (Engenheiros)                   │
├─────────────────┴───────────────────────────────────┤
│            Storage Centralizado (S3/Azure/GCS)      │
└─────────────────────────────────────────────────────┘
```

In [ ]:
%%sql -r setup_db
CREATE OR REPLACE DATABASE PARTNERS_LISBON_DEMO;

CREATE OR REPLACE SCHEMA PARTNERS_LISBON_DEMO.RAW;
CREATE OR REPLACE SCHEMA PARTNERS_LISBON_DEMO.ANALYTICS;
CREATE OR REPLACE SCHEMA PARTNERS_LISBON_DEMO.GOVERNANCE;

In [ ]:
%%sql -r setup_wh
CREATE OR REPLACE WAREHOUSE PARTNERS_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

CREATE OR REPLACE WAREHOUSE PARTNERS_WH_MEDIUM
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

In [ ]:
%%sql -r setup_roles
CREATE OR REPLACE ROLE ANALYST_ROLE;
CREATE OR REPLACE ROLE ENGINEER_ROLE;

GRANT USAGE ON DATABASE PARTNERS_LISBON_DEMO TO ROLE ANALYST_ROLE;
GRANT USAGE ON DATABASE PARTNERS_LISBON_DEMO TO ROLE ENGINEER_ROLE;
GRANT USAGE ON SCHEMA PARTNERS_LISBON_DEMO.RAW TO ROLE ANALYST_ROLE;
GRANT USAGE ON SCHEMA PARTNERS_LISBON_DEMO.RAW TO ROLE ENGINEER_ROLE;
GRANT USAGE ON SCHEMA PARTNERS_LISBON_DEMO.ANALYTICS TO ROLE ANALYST_ROLE;
GRANT USAGE ON SCHEMA PARTNERS_LISBON_DEMO.ANALYTICS TO ROLE ENGINEER_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA PARTNERS_LISBON_DEMO.RAW TO ROLE ANALYST_ROLE;
GRANT ALL ON SCHEMA PARTNERS_LISBON_DEMO.RAW TO ROLE ENGINEER_ROLE;
GRANT USAGE ON WAREHOUSE PARTNERS_WH TO ROLE ANALYST_ROLE;
GRANT USAGE ON WAREHOUSE PARTNERS_WH TO ROLE ENGINEER_ROLE;
GRANT USAGE ON WAREHOUSE PARTNERS_WH_MEDIUM TO ROLE ENGINEER_ROLE;

✅ **Resultado:** Em segundos criámos toda a infraestrutura — base de dados, schemas, warehouses e roles com permissões granulares.

⚡ Os warehouses estão `INITIALLY_SUSPENDED` — zero custo até serem usados. `AUTO_RESUME=TRUE` liga-os automaticamente na primeira query.

---
## 2. 📦 Carga de Dados

💡 **Conceito-chave:** O Snowflake aceita CSV, JSON, Parquet, Avro, ORC e XML — tudo sem ETL externo. É schema-on-read E schema-on-write.

🔑 **Porquê isto importa:** Os clientes não precisam de transformar dados antes de os carregar — eliminando ferramentas intermediárias e acelerando o onboarding de novas fontes.

| Formato | Semi-estruturado? | Caso de Uso Típico |
|---------|-------------------|-----------|
| CSV | ❌ | Exports de sistemas legados |
| JSON | ✅ | APIs, eventos, logs |
| Parquet | ✅ | Data lakes, analytics |
| Avro | ✅ | Streaming (Kafka) |
| ORC | ✅ | Migração de Hadoop |

In [ ]:
%%sql -r use_schema
USE WAREHOUSE PARTNERS_WH;
USE SCHEMA PARTNERS_LISBON_DEMO.RAW;

In [ ]:
%%sql -r create_customers
CREATE OR REPLACE TABLE PARTNERS_LISBON_DEMO.RAW.customers (
    customer_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    email VARCHAR(100),
    phone VARCHAR(30),
    city VARCHAR(50),
    country VARCHAR(50),
    region VARCHAR(10),
    signup_date DATE,
    customer_tier VARCHAR(10)
);

INSERT INTO PARTNERS_LISBON_DEMO.RAW.customers VALUES
(1,'Joao','Silva','joao.silva@email.pt','+351 912 345 678','Lisboa','Portugal','EMEA','2024-01-15','Gold'),
(2,'Maria','Santos','maria.santos@email.pt','+351 923 456 789','Porto','Portugal','EMEA','2024-02-20','Silver'),
(3,'Pedro','Costa','pedro.costa@email.pt','+351 934 567 890','Faro','Portugal','EMEA','2024-03-10','Bronze'),
(4,'Ana','Ferreira','ana.ferreira@email.pt','+351 945 678 901','Coimbra','Portugal','EMEA','2024-04-05','Gold'),
(5,'Miguel','Oliveira','miguel.oliveira@email.es','+34 612 345 678','Madrid','Spain','EMEA','2024-01-25','Silver'),
(6,'Sofia','Rodriguez','sofia.rodriguez@email.es','+34 623 456 789','Barcelona','Spain','EMEA','2024-05-12','Gold'),
(7,'Hans','Mueller','hans.mueller@email.de','+49 151 234 5678','Berlin','Germany','EMEA','2024-02-28','Bronze'),
(8,'Pierre','Dupont','pierre.dupont@email.fr','+33 612 345 678','Paris','France','EMEA','2024-06-01','Silver'),
(9,'John','Smith','john.smith@email.com','+1 212 345 6789','New York','United States','AMER','2024-03-15','Gold'),
(10,'Sarah','Johnson','sarah.johnson@email.com','+1 310 456 7890','Los Angeles','United States','AMER','2024-04-20','Silver'),
(11,'Carlos','Mendes','carlos.mendes@email.pt','+351 956 789 012','Braga','Portugal','EMEA','2024-07-01','Bronze'),
(12,'Isabella','Rossi','isabella.rossi@email.it','+39 331 234 5678','Milan','Italy','EMEA','2024-05-18','Gold'),
(13,'Yuki','Tanaka','yuki.tanaka@email.jp','+81 90 1234 5678','Tokyo','Japan','APJ','2024-06-15','Silver'),
(14,'Wei','Chen','wei.chen@email.cn','+86 138 1234 5678','Shanghai','China','APJ','2024-07-10','Gold'),
(15,'Emma','Wilson','emma.wilson@email.co.uk','+44 7911 123456','London','United Kingdom','EMEA','2024-08-01','Bronze');

In [ ]:
%%sql -r create_products
CREATE OR REPLACE TABLE PARTNERS_LISBON_DEMO.RAW.products (
    product_id INT,
    product_name VARCHAR(100),
    category VARCHAR(50),
    subcategory VARCHAR(50),
    unit_price DECIMAL(10,2),
    cost_price DECIMAL(10,2),
    description VARCHAR(200)
);

INSERT INTO PARTNERS_LISBON_DEMO.RAW.products VALUES
(101,'Laptop Pro 15','Electronics','Laptops',1299.99,850.00,'High-performance laptop with 15-inch display'),
(102,'Laptop Air 13','Electronics','Laptops',999.99,650.00,'Ultra-thin laptop for professionals'),
(103,'Wireless Mouse Elite','Accessories','Mice',79.99,35.00,'Ergonomic wireless mouse'),
(104,'Mechanical Keyboard Pro','Accessories','Keyboards',149.99,65.00,'RGB mechanical keyboard'),
(105,'Monitor UltraWide 34','Electronics','Monitors',699.99,420.00,'Ultra-wide curved monitor'),
(106,'USB-C Hub 7-in-1','Accessories','Hubs',59.99,22.00,'Multi-port USB-C hub'),
(107,'Noise-Cancelling Headphones','Audio','Headphones',349.99,180.00,'Premium wireless headphones'),
(108,'Webcam HD 4K','Electronics','Cameras',129.99,55.00,'4K webcam with auto-focus'),
(109,'External SSD 1TB','Storage','Drives',119.99,60.00,'Portable SSD USB 3.2'),
(110,'Tablet Pro 11','Electronics','Tablets',899.99,580.00,'11-inch tablet with stylus'),
(111,'Wireless Charger Pad','Accessories','Chargers',39.99,12.00,'Fast wireless charger'),
(112,'Bluetooth Speaker','Audio','Speakers',89.99,38.00,'Portable waterproof speaker');

In [ ]:
%%sql -r create_orders
CREATE OR REPLACE TABLE PARTNERS_LISBON_DEMO.RAW.orders (
    order_id INT,
    customer_id INT,
    order_date DATE,
    order_status VARCHAR(20),
    shipping_method VARCHAR(20),
    total_amount DECIMAL(10,2)
);

INSERT INTO PARTNERS_LISBON_DEMO.RAW.orders VALUES
(1001,1,'2025-01-05','Delivered','Express',1379.98),
(1002,2,'2025-01-12','Delivered','Standard',999.99),
(1003,3,'2025-01-20','Delivered','Express',229.98),
(1004,4,'2025-02-01','Delivered','Standard',1299.99),
(1005,5,'2025-02-10','Delivered','Express',449.98),
(1006,6,'2025-02-18','Delivered','Standard',699.99),
(1007,7,'2025-02-25','Delivered','Express',149.99),
(1008,8,'2025-03-05','Delivered','Standard',1899.98),
(1009,9,'2025-03-12','Delivered','Express',349.99),
(1010,10,'2025-03-20','Delivered','Standard',239.98),
(1011,1,'2025-03-28','Delivered','Express',899.99),
(1012,11,'2025-04-05','Delivered','Standard',79.99),
(1013,12,'2025-04-12','Delivered','Express',1349.98),
(1014,13,'2025-04-20','Shipped','Standard',469.98),
(1015,14,'2025-04-28','Shipped','Express',2199.98),
(1016,2,'2025-05-05','Shipped','Standard',129.99),
(1017,4,'2025-05-12','Shipped','Express',349.99),
(1018,6,'2025-05-20','Processing','Standard',1599.98),
(1019,9,'2025-05-28','Processing','Express',179.98),
(1020,15,'2025-06-05','Processing','Standard',899.99),
(1021,1,'2025-06-10','Processing','Express',259.98),
(1022,3,'2025-06-15','Pending','Standard',1299.99),
(1023,5,'2025-06-20','Pending','Express',89.99),
(1024,7,'2025-06-25','Pending','Standard',449.98),
(1025,8,'2025-06-30','Pending','Express',119.99),
(1026,10,'2025-07-01','Pending','Standard',699.99),
(1027,12,'2025-07-02','Pending','Express',39.99),
(1028,14,'2025-07-03','Pending','Standard',1299.99),
(1029,11,'2025-07-04','Pending','Express',209.98),
(1030,15,'2025-07-05','Pending','Standard',349.99);

In [ ]:
%%sql -r create_order_items
CREATE OR REPLACE TABLE PARTNERS_LISBON_DEMO.RAW.order_items (
    item_id INT,
    order_id INT,
    product_id INT,
    quantity INT,
    unit_price DECIMAL(10,2),
    discount_pct DECIMAL(5,2)
);

INSERT INTO PARTNERS_LISBON_DEMO.RAW.order_items VALUES
(1,1001,101,1,1299.99,0),
(2,1001,103,1,79.99,0),
(3,1002,102,1,999.99,0),
(4,1003,104,1,149.99,0),
(5,1003,103,1,79.99,0),
(6,1004,101,1,1299.99,0),
(7,1005,107,1,349.99,0),
(8,1005,106,1,59.99,10),
(9,1006,105,1,699.99,0),
(10,1007,104,1,149.99,0),
(11,1008,101,1,1299.99,5),
(12,1008,105,1,699.99,0),
(13,1009,107,1,349.99,0),
(14,1010,103,2,79.99,0),
(15,1010,111,2,39.99,5),
(16,1011,110,1,899.99,0),
(17,1012,103,1,79.99,0),
(18,1013,101,1,1299.99,0),
(19,1013,111,1,39.99,10),
(20,1014,107,1,349.99,0),
(21,1014,109,1,119.99,0),
(22,1015,101,1,1299.99,0),
(23,1015,110,1,899.99,0),
(24,1016,108,1,129.99,0),
(25,1017,107,1,349.99,0),
(26,1018,102,1,999.99,5),
(27,1018,105,1,699.99,0),
(28,1019,106,2,59.99,0),
(29,1019,111,1,39.99,10),
(30,1020,110,1,899.99,0),
(31,1021,104,1,149.99,0),
(32,1021,109,1,119.99,0),
(33,1022,101,1,1299.99,0),
(34,1023,112,1,89.99,0),
(35,1024,107,1,349.99,5),
(36,1024,106,1,59.99,10),
(37,1025,109,1,119.99,0),
(38,1026,105,1,699.99,0),
(39,1027,111,1,39.99,0),
(40,1028,101,1,1299.99,0),
(41,1029,103,1,79.99,0),
(42,1029,108,1,129.99,0),
(43,1030,107,1,349.99,0),
(44,1005,112,1,89.99,0),
(45,1008,106,1,59.99,0),
(46,1013,106,1,59.99,0),
(47,1015,104,1,149.99,5),
(48,1018,108,1,129.99,0),
(49,1024,112,1,89.99,0),
(50,1030,111,1,39.99,0);

In [ ]:
%%sql -r create_reviews
CREATE OR REPLACE TABLE PARTNERS_LISBON_DEMO.RAW.product_reviews (
    review_id INT,
    product_id INT,
    customer_id INT,
    review_date DATE,
    rating INT,
    review_text VARCHAR(500)
);

INSERT INTO PARTNERS_LISBON_DEMO.RAW.product_reviews VALUES
(1,101,1,'2025-01-20',5,'Excellent laptop, very fast and the display is amazing. Best purchase I made this year.'),
(2,102,2,'2025-01-25',4,'Great thin laptop, perfect for travel. Battery life could be slightly better.'),
(3,103,3,'2025-02-05',3,'Decent mouse but nothing special. Works fine for daily use.'),
(4,104,4,'2025-02-15',5,'Love the mechanical feel and RGB lighting. Very satisfying to type on.'),
(5,105,6,'2025-03-01',5,'The ultra-wide monitor is incredible for productivity. Highly recommend.'),
(6,107,9,'2025-03-15',4,'Great noise cancellation. Sound quality is premium. Comfortable for long sessions.'),
(7,101,12,'2025-04-15',5,'Powerful machine, handles everything I throw at it. Worth every penny.'),
(8,110,1,'2025-04-01',4,'Nice tablet, stylus works great. Wish it had more storage options.'),
(9,106,8,'2025-03-10',2,'Hub stopped working after a month. Disappointing quality for the price.'),
(10,109,10,'2025-03-25',4,'Fast and reliable SSD. Transfer speeds are impressive.'),
(11,108,14,'2025-05-01',3,'Webcam is okay, 4K quality is good but software is buggy.'),
(12,111,5,'2025-02-15',1,'Charger overheated and stopped working in two weeks. Terrible product.'),
(13,112,7,'2025-03-05',4,'Good sound for the size. Waterproof design is great for outdoor use.'),
(14,107,4,'2025-05-15',5,'Second pair I bought - giving one as a gift. Absolutely love them.'),
(15,102,15,'2025-06-10',4,'Solid laptop for the price. Fast boot times and nice keyboard.');

In [ ]:
%%sql -r verify_load
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM PARTNERS_LISBON_DEMO.RAW.customers
UNION ALL
SELECT 'products', COUNT(*) FROM PARTNERS_LISBON_DEMO.RAW.products
UNION ALL
SELECT 'orders', COUNT(*) FROM PARTNERS_LISBON_DEMO.RAW.orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM PARTNERS_LISBON_DEMO.RAW.order_items
UNION ALL
SELECT 'product_reviews', COUNT(*) FROM PARTNERS_LISBON_DEMO.RAW.product_reviews;

In [ ]:
import matplotlib.pyplot as plt

tables = verify_load['TABLE_NAME'].tolist()
counts = verify_load['ROW_COUNT'].tolist()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(tables, counts, color=['#29B5E8', '#11567F', '#7D44CF', '#FF9F36', '#71D3DC'])
ax.set_title('Contagem de Registos por Tabela', fontsize=14, fontweight='bold')
ax.set_xlabel('Tabela')
ax.set_ylabel('Número de Registos')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

✅ **Resultado:** 5 tabelas criadas e carregadas com dados realistas — clientes de 9 países, 12 produtos, 30 encomendas.

💡 **Nota importante:** Usámos INSERT para a demo, mas em produção usaríamos:
- `COPY INTO` para cargas batch de ficheiros
- `Snowpipe` para ingestão contínua (que vemos a seguir!)
- `Snowpipe Streaming` para real-time via SDK

---
## 3. 🔄 Snowpipe — Ingestão Contínua

💡 **Conceito-chave:** O Snowpipe monitoriza ficheiros novos num stage e carrega-os automaticamente — zero intervenção humana, zero orquestração.

🔑 **Porquê isto importa:** Elimina a complexidade de manter pipelines de ingestão (cron jobs, Airflow, scripts custom). Os dados ficam disponíveis em ~1 minuto após chegarem ao stage.

| Método | Latência | Orquestração | Caso de Uso |
|--------|----------|--------------|-------------|
| COPY INTO | Minutos | Manual/Schedule | Batch tradicional |
| Snowpipe | ~1 min | Automática (notificações S3/Azure/GCS) | Streaming de ficheiros |
| Snowpipe Streaming | <10s | SDK (Java/Python) | Real-time de baixa latência |

⚡ **Nota:** Snowpipe usa compute serverless — paga-se apenas pelo tempo de ingestão, sem warehouse dedicado.

In [ ]:
%%sql -r snowpipe_stage
CREATE OR REPLACE STAGE PARTNERS_LISBON_DEMO.RAW.partners_ingest_stage
    FILE_FORMAT = (TYPE = 'JSON');

In [ ]:
%%sql -r snowpipe_table
CREATE OR REPLACE TABLE PARTNERS_LISBON_DEMO.RAW.raw_events (
    event_id INT,
    event_type VARCHAR(50),
    event_timestamp TIMESTAMP_NTZ,
    user_id INT,
    event_data VARIANT,
    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

In [ ]:
%%sql -r snowpipe_create
CREATE OR REPLACE PIPE PARTNERS_LISBON_DEMO.RAW.partners_event_pipe
    AUTO_INGEST = FALSE
AS
COPY INTO PARTNERS_LISBON_DEMO.RAW.raw_events (event_id, event_type, event_timestamp, user_id, event_data)
FROM (
    SELECT
        $1:event_id::INT,
        $1:event_type::VARCHAR,
        $1:event_timestamp::TIMESTAMP_NTZ,
        $1:user_id::INT,
        $1:event_data::VARIANT
    FROM @PARTNERS_LISBON_DEMO.RAW.partners_ingest_stage
);

👇 A seguir geramos 20 eventos JSON sintéticos e colocamo-los no stage. Em produção, estes ficheiros chegariam via notificação S3 ou Azure Event Grid, e o Snowpipe carregá-los-ia automaticamente.

In [ ]:
%%sql -r snowpipe_generate
COPY INTO @PARTNERS_LISBON_DEMO.RAW.partners_ingest_stage/events/batch_001.json
FROM (
    SELECT OBJECT_CONSTRUCT(
        'event_id', SEQ4(),
        'event_type', CASE MOD(SEQ4(), 4)
            WHEN 0 THEN 'page_view'
            WHEN 1 THEN 'add_to_cart'
            WHEN 2 THEN 'purchase'
            WHEN 3 THEN 'search'
        END,
        'event_timestamp', DATEADD('minute', -SEQ4() * 15, CURRENT_TIMESTAMP()),
        'user_id', MOD(SEQ4(), 15) + 1,
        'event_data', OBJECT_CONSTRUCT(
            'page', '/products/' || (MOD(SEQ4(), 12) + 101),
            'session_id', UUID_STRING(),
            'device', CASE MOD(SEQ4(), 3) WHEN 0 THEN 'mobile' WHEN 1 THEN 'desktop' ELSE 'tablet' END
        )
    ) AS json_data
    FROM TABLE(GENERATOR(ROWCOUNT => 20))
);

In [ ]:
%%sql -r snowpipe_refresh
ALTER PIPE PARTNERS_LISBON_DEMO.RAW.partners_event_pipe REFRESH;

In [ ]:
%%sql -r snowpipe_status
SELECT SYSTEM$PIPE_STATUS('PARTNERS_LISBON_DEMO.RAW.partners_event_pipe');

In [ ]:
%%sql -r snowpipe_count
SELECT COUNT(*) AS event_count FROM PARTNERS_LISBON_DEMO.RAW.raw_events;

✅ **Resultado:** Pipeline de ingestão completo em funcionamento:
1. Stage para receber ficheiros
2. Pipe com transformação (JSON → tabela relacional)
3. Dados carregados automaticamente

🔑 **Em produção:** Configuramos uma notificação S3/Azure/GCS → o Snowpipe carrega automaticamente cada ficheiro novo em ~1 minuto. **Zero orquestração, zero manutenção.**

---
## 4. 🔒 Governança de Dados — O Momento WOW!

💡 **Conceito-chave:** A governança está nos **DADOS**, não na aplicação. Masking policies e row access policies são aplicadas automaticamente — a aplicação nem sabe que existem.

🔑 **Porquê isto importa:** Com uma única policy SQL, garantimos que dados sensíveis ficam protegidos para TODAS as ferramentas de acesso (BI, SQL, APIs, Python) — sem alterações de código. Conformidade GDPR/LGPD automática.

🔥 Vamos ver como a **MESMA query**, executada pelo **MESMO utilizador**, retorna dados **DIFERENTES** baseado no role ativo.

In [ ]:
%%sql -r mask_email
CREATE OR REPLACE MASKING POLICY PARTNERS_LISBON_DEMO.GOVERNANCE.mask_email AS
(val STRING) RETURNS STRING ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'ENGINEER_ROLE') THEN val
        ELSE REGEXP_REPLACE(val, '.+@', '****@')
    END;

ALTER TABLE PARTNERS_LISBON_DEMO.RAW.customers
    MODIFY COLUMN email SET MASKING POLICY PARTNERS_LISBON_DEMO.GOVERNANCE.mask_email;

In [ ]:
%%sql -r mask_phone
CREATE OR REPLACE MASKING POLICY PARTNERS_LISBON_DEMO.GOVERNANCE.mask_phone AS
(val STRING) RETURNS STRING ->
    CASE
        WHEN CURRENT_ROLE() IN ('ACCOUNTADMIN', 'ENGINEER_ROLE') THEN val
        ELSE CONCAT(LEFT(val, 4), ' *** *** ***')
    END;

ALTER TABLE PARTNERS_LISBON_DEMO.RAW.customers
    MODIFY COLUMN phone SET MASKING POLICY PARTNERS_LISBON_DEMO.GOVERNANCE.mask_phone;

In [ ]:
%%sql -r row_access
CREATE OR REPLACE ROW ACCESS POLICY PARTNERS_LISBON_DEMO.GOVERNANCE.region_access_policy AS
(region_val VARCHAR) RETURNS BOOLEAN ->
    CASE
        WHEN CURRENT_ROLE() = 'ACCOUNTADMIN' THEN TRUE
        WHEN CURRENT_ROLE() = 'ANALYST_ROLE' AND region_val = 'EMEA' THEN TRUE
        ELSE FALSE
    END;

ALTER TABLE PARTNERS_LISBON_DEMO.RAW.customers
    ADD ROW ACCESS POLICY PARTNERS_LISBON_DEMO.GOVERNANCE.region_access_policy ON (region);

### 👁️ Visibilidade como ACCOUNTADMIN (acesso total)

👁️ Como ACCOUNTADMIN, temos visibilidade total sobre todos os dados — emails completos, telefones completos, todas as regiões. No próximo passo veremos como isto muda com roles diferentes.

In [ ]:
%%sql -r gov_full_view
SELECT customer_id, first_name, last_name, email, phone, city, country, region
FROM PARTNERS_LISBON_DEMO.RAW.customers
LIMIT 5;

### 📊 Contagem por região

👁️ Como ACCOUNTADMIN vemos as 3 regiões (EMEA: 11, AMER: 2, APJ: 2). Se executássemos como ANALYST_ROLE, veríamos **APENAS EMEA** — a row access policy filtra automaticamente.

In [ ]:
%%sql -r gov_region_counts
SELECT region, COUNT(*) AS customer_count
FROM PARTNERS_LISBON_DEMO.RAW.customers
GROUP BY region
ORDER BY customer_count DESC;

### 🎯 Comparação de Visibilidade por Role

| | 👑 ACCOUNTADMIN | 📊 ANALYST_ROLE |
|---|---|---|
| **Email** | joao.silva@email.pt | ****@email.pt |
| **Telefone** | +351 912 345 678 | +351 *** *** *** |
| **Regiões visíveis** | EMEA, AMER, APJ (15 registos) | Apenas EMEA (11 registos) |
| **Dados** | Tudo visível | Apenas a sua região |

✅ **Resultado:** A mesma tabela, a mesma query, mas resultados completamente diferentes baseados no role. **Zero código na aplicação.**

💡 **Conceito-chave:** A governança está nos DADOS, não na app. Qualquer ferramenta (Tableau, Power BI, Python, dbt) respeita estas policies automaticamente.

---
## 5. ⚡ Performance & Gestão de Custos

💡 **Conceito-chave:** O modelo pay-per-second com auto-suspend garante que só pagamos quando estamos a usar compute. Controlo total sobre custos.

🔑 **Porquê isto importa:** Os clientes preocupam-se com custos imprevisíveis na cloud. O Snowflake dá-lhes visibilidade e controlo granular.

| Tamanho | Créditos/h | Nós | Caso de Uso |
|---------|------------|-----|-------------|
| XS | 1 | 1 | Dev, queries simples |
| S | 2 | 2 | BI, relatórios |
| M | 4 | 4 | ETL, analytics |
| L | 8 | 8 | Cargas pesadas |
| XL+ | 16-512 | 16-512 | Big data, ML |

⚡ **3 níveis de cache** (do mais rápido ao mais lento):
1. 🟢 **Result Cache** — Resultado idêntico em <100ms (24h)
2. 🟡 **Local Disk Cache** — SSD do warehouse (dados quentes)
3. 🔵 **Remote Disk Cache** — Storage centralizado

In [ ]:
%%sql -r revenue_query
SELECT
    p.category,
    p.subcategory,
    COUNT(DISTINCT o.order_id) AS num_orders,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)) AS total_revenue,
    AVG(oi.unit_price) AS avg_unit_price
FROM PARTNERS_LISBON_DEMO.RAW.orders o
JOIN PARTNERS_LISBON_DEMO.RAW.order_items oi ON o.order_id = oi.order_id
JOIN PARTNERS_LISBON_DEMO.RAW.products p ON oi.product_id = p.product_id
GROUP BY p.category, p.subcategory
ORDER BY total_revenue DESC;

In [ ]:
import matplotlib.pyplot as plt

df = revenue_query.sort_values('TOTAL_REVENUE', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#29B5E8' if cat == 'Electronics' else '#7D44CF' if cat == 'Audio' else '#FF9F36' if cat == 'Accessories' else '#71D3DC'
          for cat in df['CATEGORY']]
bars = ax.barh(df['SUBCATEGORY'], df['TOTAL_REVENUE'], color=colors)
ax.set_title('Receita por Subcategoria', fontsize=14, fontweight='bold')
ax.set_xlabel('Receita Total (€)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'€{x:,.0f}'))
plt.tight_layout()
plt.show()

### 🚀 Demonstração de Cache

💡 **Conceito-chave:** O Result Cache guarda resultados de queries idênticas durante 24h. Execute a query abaixo **DUAS VEZES** — a segunda execução será **instantânea** (~0ms) porque reutiliza o resultado em cache.

▶️ **Exercício:** Executar → ver tempo → executar novamente → ver tempo

In [ ]:
%%sql -r cache_demo
SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)) AS total_revenue
FROM PARTNERS_LISBON_DEMO.RAW.orders o
JOIN PARTNERS_LISBON_DEMO.RAW.order_items oi ON o.order_id = oi.order_id
JOIN PARTNERS_LISBON_DEMO.RAW.products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;

### 📊 Tamanhos de Warehouse & Créditos

In [ ]:
import matplotlib.pyplot as plt

sizes = ['XS', 'S', 'M', 'L', 'XL', '2XL', '3XL', '4XL']
credits = [1, 2, 4, 8, 16, 32, 64, 128]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(sizes, credits, color='#29B5E8', edgecolor='#11567F', linewidth=1.2)
ax.set_title('Créditos por Hora vs Tamanho do Warehouse', fontsize=14, fontweight='bold')
ax.set_xlabel('Tamanho do Warehouse')
ax.set_ylabel('Créditos / Hora')
ax.set_yscale('log', base=2)
ax.set_yticks(credits)
ax.set_yticklabels([str(c) for c in credits])
for bar, credit in zip(bars, credits):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.1,
            str(credit), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
%%sql -r resource_monitor
CREATE OR REPLACE RESOURCE MONITOR partners_monitor
    WITH CREDIT_QUOTA = 100
    FREQUENCY = MONTHLY
    START_TIMESTAMP = IMMEDIATELY
    TRIGGERS
        ON 75 PERCENT DO NOTIFY
        ON 90 PERCENT DO NOTIFY
        ON 100 PERCENT DO SUSPEND;

✅ **Resultado:**
- Query analítica com JOIN de 3 tabelas — resposta sub-segundo
- Result Cache — segunda execução instantânea
- Resource Monitor — controlo automático de custos

💡 **Boas práticas de custo:**

| Prática | Descrição |
|---------|-----------|
| **Auto-Suspend 60s** | Desliga warehouse após 60s inativo |
| **Result Cache** | Reutilização automática (24h, grátis) |
| **Multi-cluster** | Escalar para concorrência, não para velocidade |
| **Resource Monitors** | Alertas em 75%, suspend em 100% |
| **Right-sizing** | XS para dev, M/L para produção |

---
## 🎯 Resumo — O que demonstrámos

| Capacidade | Valor para o Cliente |
|------------|---------------------|
| 🏗️ Setup instantâneo | Infraestrutura em segundos, sem tickets |
| 📦 Data Loading | Suporte nativo a múltiplos formatos |
| 🔄 Snowpipe | Ingestão contínua sem orquestração |
| 🔒 Governança | Proteção ao nível dos dados, não da app |
| ⚡ Performance | Caching automático, scaling elástico |
| 💰 Custos | Pay-per-use + Resource Monitors |

---

🔑 **Mensagens-chave:**
1. **Separação compute/storage** = escalar sem limites
2. **Governança nativa** = conformidade sem código
3. **Pay-per-second** = custo previsível e controlável
4. **Zero orquestração** = menos infraestrutura para gerir

**👉 Próximo:** Aprofundamento II — IA, Pipelines Declarativos e Aplicações!